# GANs on a CPU

**Day 9 · Human and Machine Perception**

A generative adversarial network (GAN) trains two networks against each other. The **generator** turns random noise into samples. The **discriminator** tries to tell real samples from generated ones. Each one gets better by beating the other.

This notebook is built to run on a plain CPU, so it needs no GPU:

| Part | Data | Training time on a CPU |
|---|---|---|
| 1 | 2D points on a ring | about 10 seconds |
| 2 | MNIST digits | under a minute (3 epochs) |

**How to use it:** run the cells top to bottom. Each part ends with **Try it** questions.

## Setup

Needs `torch`, `torchvision` and `matplotlib`, which Colab already has. MNIST downloads into `data/` the first time you run Part 2.

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(0)
np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on", device)

---
## The idea and the math

The discriminator $D(x)$ outputs the probability that $x$ is real. The generator $G(z)$ maps noise $z \sim \mathcal{N}(0, I)$ to a sample. They play a min-max game (Goodfellow et al., 2014):

$$\min_G \max_D \; \mathbb{E}_{x \sim p_\text{data}}\big[\log D(x)\big] + \mathbb{E}_{z \sim \mathcal{N}(0,I)}\big[\log\big(1 - D(G(z))\big)\big]$$

- $D$ wants $D(x) \to 1$ on real data and $D(G(z)) \to 0$ on fakes.
- $G$ wants $D(G(z)) \to 1$, so its fakes pass as real.

In practice the generator maximizes $\log D(G(z))$ instead of minimizing $\log(1 - D(G(z)))$. It is the same goal, but the gradient is stronger early on, when the fakes are easy to spot.

Each training step has two halves:
1. **Discriminator step:** show it a real batch (label 1) and a fake batch (label 0), and update $D$ only.
2. **Generator step:** make a fresh fake batch, label it 1, and update $G$ only, through $D$.

`train_gan` below does exactly this for any generator, discriminator and source of real data. `D` outputs a raw score (a logit). `BCEWithLogitsLoss` applies the sigmoid for us.

In [ ]:
def train_gan(G, D, real_batches, steps, z_dim, lr=2e-4, snapshot_at=(), fixed_z=None):
    # real_batches(): returns one batch of real data. Returns the loss history and samples from fixed_z at each snapshot step.
    G, D = G.to(device), D.to(device)
    opt_G = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    bce = nn.BCEWithLogitsLoss()
    history, snapshots = {"D": [], "G": []}, {}
    for step in range(1, steps + 1):
        x = real_batches().to(device)
        n = x.shape[0]
        ones, zeros = torch.ones(n, 1, device=device), torch.zeros(n, 1, device=device)

        # 1. discriminator step: real -> 1, fake -> 0
        fake = G(torch.randn(n, z_dim, device=device)).detach()   # detach: do not update G here
        loss_D = bce(D(x), ones) + bce(D(fake), zeros)
        opt_D.zero_grad(); loss_D.backward(); opt_D.step()

        # 2. generator step: make D call the fakes real
        fake = G(torch.randn(n, z_dim, device=device))
        loss_G = bce(D(fake), ones)
        opt_G.zero_grad(); loss_G.backward(); opt_G.step()

        history["D"].append(loss_D.item()); history["G"].append(loss_G.item())
        if step in snapshot_at and fixed_z is not None:
            G.eval()                      # batch norm: use running statistics when sampling
            with torch.no_grad():
                snapshots[step] = G(fixed_z.to(device)).cpu()
            G.train()
    return history, snapshots


def plot_losses(history, every=1, title="Losses"):
    plt.figure(figsize=(7, 3))
    for k in history:
        h = np.convolve(history[k], np.ones(every) / every, mode="valid")
        plt.plot(h, label=f"{k} loss")
    plt.xlabel("training step"); plt.legend(); plt.title(title); plt.show()

---
## Part 1. A GAN on 2D points

Real data: points from 8 small clusters on a ring. With only two numbers per sample we can **see** everything: the real data, the fakes, and how the generator moves them during training.

A good generator puts points on **all 8** clusters. A common failure is **mode collapse**: the generator finds a few clusters that fool the discriminator and stops covering the rest.

In [ ]:
N_MODES, RADIUS, SPREAD = 8, 2.0, 0.05
angles = torch.arange(N_MODES) * 2 * np.pi / N_MODES
centers = torch.stack([RADIUS * torch.cos(angles), RADIUS * torch.sin(angles)], dim=1)

def ring_batch(n=256):
    idx = torch.randint(0, N_MODES, (n,))
    return centers[idx] + SPREAD * torch.randn(n, 2)

def modes_covered(samples, min_share=0.02):
    # a mode counts as covered if at least 2% of samples land within 3 standard deviations of its center
    d = torch.cdist(samples, centers)
    near = d.min(dim=1).values < 3 * SPREAD
    which = d.argmin(dim=1)[near]
    counts = torch.bincount(which, minlength=N_MODES).float() / len(samples)
    return int((counts >= min_share).sum()), near.float().mean().item()

real = ring_batch(2000)
plt.figure(figsize=(4, 4)); plt.scatter(real[:, 0], real[:, 1], s=2)
plt.axis("equal"); plt.title("Real data: 8 clusters on a ring"); plt.show()

In [ ]:
Z2 = 2
def make_2d_gan(width=64):
    G = nn.Sequential(nn.Linear(Z2, width), nn.ReLU(), nn.Linear(width, width), nn.ReLU(), nn.Linear(width, 2))
    D = nn.Sequential(nn.Linear(2, width), nn.ReLU(), nn.Linear(width, width), nn.ReLU(), nn.Linear(width, 1))
    return G, D

torch.manual_seed(0)
G2, D2 = make_2d_gan()
fixed_z2 = torch.randn(1000, Z2)
snaps = (1, 250, 500, 1000, 2000, 3000)

t0 = time.time()
hist2, shots2 = train_gan(G2, D2, ring_batch, steps=3000, z_dim=Z2, lr=1e-3, snapshot_at=snaps, fixed_z=fixed_z2)
print(f"trained in {time.time() - t0:.1f} s")

fig, axes = plt.subplots(1, len(snaps), figsize=(3 * len(snaps), 3))
for ax, s in zip(axes, snaps):
    ax.scatter(real[:, 0], real[:, 1], s=1, c="lightgray")
    ax.scatter(shots2[s][:, 0], shots2[s][:, 1], s=2, c="crimson")
    m, q = modes_covered(shots2[s])
    ax.set_title(f"step {s}\n{m}/8 modes, {q:.0%} on target", fontsize=9)
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal"); ax.axis("off")
plt.suptitle("Gray: real data. Red: generator samples (same noise every time)"); plt.tight_layout(); plt.show()

plot_losses(hist2, every=50, title="2D GAN losses (moving average)")

### Try it (Part 1)

Each run takes seconds, so change one thing at a time and re-run the training cell.

1. **Read the losses.** Do they go to zero, or do they settle? Why can't both players win at once?
2. **Mode collapse.** Re-run with `torch.manual_seed(1)`, `2` and `3`. How many of the 8 modes does each run cover? Does a lower generator loss mean better coverage?
3. **An unfair game.** Make the discriminator weak with `make_2d_gan(width=8)`, or give it a much higher learning rate than the generator. What happens to the samples?
4. **Noise size.** The generator gets 2 numbers of noise (`Z2 = 2`). Try `Z2 = 1`. Can a 1-number input still cover a ring of 8 clusters? Look at the shape the samples form.

---
## Part 2. A GAN on MNIST digits

Same training loop, now on 28×28 images. Both networks are small fully connected networks, so this stays fast on a CPU. The generator uses batch normalization, which helps it train in only a few epochs. The generator ends with `tanh`, so pixels are scaled to [-1, 1].

We keep one fixed batch of noise and decode it after every epoch, so you can watch the **same** noise vectors turn into digits.

In [ ]:
tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])   # pixels to [-1, 1]
mnist = datasets.MNIST("data", train=True, download=True, transform=tfm)
loader = DataLoader(mnist, batch_size=64, shuffle=True, drop_last=True)

def endless(loader):
    while True:
        for x, _ in loader:
            yield x.view(x.shape[0], -1)

ZM = 64
Gm = nn.Sequential(nn.Linear(ZM, 256), nn.BatchNorm1d(256), nn.LeakyReLU(0.2),
                   nn.Linear(256, 512), nn.BatchNorm1d(512), nn.LeakyReLU(0.2),
                   nn.Linear(512, 28 * 28), nn.Tanh())
Dm = nn.Sequential(nn.Linear(28 * 28, 512), nn.LeakyReLU(0.2),
                   nn.Linear(512, 256), nn.LeakyReLU(0.2),
                   nn.Linear(256, 1))

def show_digits(x, title, n_row=2):
    x = (x.view(-1, 28, 28) + 1) / 2
    n = x.shape[0]; per = n // n_row
    grid = torch.cat([torch.cat(list(x[r * per:(r + 1) * per]), dim=1) for r in range(n_row)], dim=0)
    plt.figure(figsize=(per * 0.8, n_row * 0.8)); plt.imshow(grid, cmap="gray"); plt.axis("off"); plt.title(title); plt.show()

In [ ]:
EPOCHS = 3                      # about 10-20 seconds per epoch on a CPU
steps_per_epoch = len(loader)
torch.manual_seed(0)
fixed_zm = torch.randn(16, ZM)
batches = endless(loader)

t0 = time.time()
histm, shotsm = train_gan(Gm, Dm, lambda: next(batches), steps=EPOCHS * steps_per_epoch, z_dim=ZM, lr=5e-4,
                          snapshot_at={e * steps_per_epoch for e in range(1, EPOCHS + 1)} | {100}, fixed_z=fixed_zm)
print(f"trained {EPOCHS} epochs in {time.time() - t0:.0f} s")

for s in sorted(shotsm):
    show_digits(shotsm[s], f"step {s} (epoch {s / steps_per_epoch:.2f})")
plot_losses(histm, every=100, title="MNIST GAN losses (moving average)")

### Walk through the noise space

Pick two noise vectors and decode the points on the straight line between them. If the generator has learned a smooth space, the digit should morph gradually instead of jumping.

In [ ]:
torch.manual_seed(3)
z_a, z_b = torch.randn(ZM), torch.randn(ZM)
ts = torch.linspace(0, 1, 10)
Gm.eval()
with torch.no_grad():
    walk = Gm(torch.stack([(1 - t) * z_a + t * z_b for t in ts]).to(device)).cpu()
show_digits(walk, "From noise A to noise B", n_row=1)

### Try it (Part 2)

1. **Sharp but strange.** If a classmate did Track A, compare these samples to their VAE samples. Which are sharper? Which look more like real digits? Why might a GAN be sharper than a VAE?
2. **Which digits?** Count the digit classes in the last grid. Are all ten there? Is this the same failure as mode collapse in Part 1?
3. **Is the discriminator fooled?** Run `Gm.eval(); torch.sigmoid(Dm(Gm(fixed_zm.to(device)))).mean()`. What does the discriminator think of the fakes? Do you agree with it?
4. **Perception link.** A GAN is trained only to fool another network. What would it take for its digits to fool a *person*? How would you test that in an experiment?
5. **If you have time:** set `EPOCHS = 6` and re-run from the cell that builds `Gm` and `Dm`. Does it keep improving?

---
**Reference.** Goodfellow, Pouget-Abadie, Mirza, Xu, Warde-Farley, Ozair, Courville and Bengio. *Generative Adversarial Nets.* NeurIPS (2014).